In [2]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt #matplotlib绘图库
import copy #python的copy模块提供了浅拷贝和深拷贝操作
import os
import sys

print("Load a ply point cloud, print it, and render it")
pcd = o3d.io.read_point_cloud(r"C:\Users\Jin Cai\Desktop\open3D_tutorial\data\extract\DemoCropPointCloud\fragment.ply")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Load a ply point cloud, print it, and render it


### Plane segmentation 平面分割

**注意：每次用`segment_plane`只能提取最大的平面，可以通过迭代提取不同的平面**  

Open3D also supports segmentation of geometric primitives from point clouds using RANSAC. To find the plane with the largest support in the point cloud, we can use `segment_plane`. The method has three arguments: `distance_threshold` defines the maximum distance a point can have to an estimated plane to be considered an inlier, `ransac_n` defines the number of points that are randomly sampled to estimate a plane, and `num_iterations` defines how often a random plane is sampled and verified.
The function then returns the plane as (a, b, c, d)such that for each point (x, y, z)on the plane we have
ax + by + cz + d = 0.  
The function further returns a list of indices of the inlier points.

Open3D 也支持使用 RANSAC 从点云中分割几何基本体。为了找到点云中具有最多支持点的平面，我们可以使用`segment_plane`方法。  
该方法有三个参数：
- `distance_threshold`定义一个点距离估计平面的最大允许距离。如果点到平面的距离小于这个阈值，那么该点就被认为是内点（inlier）。
- `ransac_n`定义每次为了估计一个平面而随机采样的点的数量。对于平面来说，通常至少需要 3 个点。
- `num_iterations`定义随机采样平面并进行验证的迭代次数。

该函数最终会返回平面的参数：(a,b,c,d)，对于位于该平面上的任意点(x,y,z)，满足：ax + by + cz + d = 0  
此外，该函数还会返回一个列表，其中包含所有被判断为该平面内点（inlier）的点的索引

In [3]:
#使用RANSAC找最大的平面
#返回两个参数：1.plane_model:平面方程中的a,b,c,d  2.inliers:该平面中的点的index
plane_model, inliers = pcd.segment_plane(
    distance_threshold=0.01, #距离小于0.01的为inliers
    ransac_n=3, #每次随机采样3个点
    num_iterations=1000  #验证1000次
)

#取出平面参数
[a, b, c, d] = plane_model
#打印平面方程，“.2f”表示保留两位小数
print(
    f"Plane equation: {a:.2f}x + {b:.2f}y + {c:.2f}z + {d:.2f} = 0"
)
#把属于平面的inliers提取出来，得到inlier_cloud
inlier_cloud = pcd.select_by_index(inliers)
#把平面涂成红色
inlier_cloud.paint_uniform_color([1.0, 0, 0])
#提取出不属于平面的outliers，注意最后的invert=True是反选的意思
outlier_cloud = pcd.select_by_index(inliers, invert=True)
o3d.visualization.draw_geometries([inlier_cloud, outlier_cloud])


Plane equation: -0.00x + 1.00y + 0.02z + -2.43 = 0
